# Explore MP AnnData
Exploration notebook for `MP_all_samples.h5ad`.

**Panel:** DAPI · CD8 · TCR · GrB · HLA-DR · PD-1 · Pan-CK · Autofluorescence  
**Samples:** 40 MP project scans  
**Cells:** Ignore-region cells removed at creation time

In [ ]:
import anndata as ad
import pandas as pd
import numpy as np

adata_path = '/vast/projects/SOLACE2/imalki/Imalki_July2022/MP_project/MP_all_samples.h5ad'
adata = ad.read_h5ad(adata_path)
print(adata)

## Basic structure

In [ ]:
print(f"Cells:    {adata.n_obs:,}")
print(f"Features: {adata.n_vars}")
print(f"Samples:  {adata.obs['sample_id'].nunique()}")
print(f"\nobs columns ({len(adata.obs.columns)}):")
print(adata.obs.columns.tolist())
print(f"\nvar names: {adata.var_names.tolist()}")
print(f"\nobsm keys: {list(adata.obsm.keys())}")
print(f"\nuns keys:  {list(adata.uns.keys())}")

In [ ]:
# Feature (marker) metadata
adata.var

In [ ]:
# Cell metadata
adata.obs.head()

## Cells per sample

In [ ]:
adata.obs['sample_id'].value_counts()

## Metadata columns

In [ ]:
# Batch
print("Batch number:")
print(adata.obs['batch_number'].value_counts())

In [ ]:
# Cancer grouping
print("Main cancer grouping:")
print(adata.obs['main_cancer_grouping'].value_counts())

In [ ]:
# Category
print("Category:")
print(adata.obs['category'].value_counts())

In [ ]:
# Type of cancer per sample
adata.obs[['sample_id', 'type_of_cancer', 'main_cancer_grouping', 'category', 'batch_number']].drop_duplicates('sample_id').sort_values('sample_id')

## Cell classification

In [ ]:
adata.obs['classification'].value_counts()

In [ ]:
# Classification breakdown per sample
adata.obs.groupby(['sample_id', 'classification']).size().unstack(fill_value=0)

## Annotations (Tumour / Stroma)

In [ ]:
adata.obs['containingannotations'].value_counts()

In [ ]:
# Confirm no Ignore cells remain
n_ignore = adata.obs['containingannotations'].str.contains(r'\[Ignore\]', na=False).sum()
print(f"Cells in Ignore regions: {n_ignore}")

## Marker intensity matrix

In [ ]:
# Convert X to DataFrame for inspection
marker_df = adata.to_df()
marker_df.describe().round(2)

In [ ]:
# Subset to a single marker
cd8_col = [v for v in adata.var_names if 'CD8' in v][0]
adata_cd8 = adata[:, cd8_col]
print(f"CD8 column: {cd8_col}")
print(f"Mean CD8 intensity: {adata_cd8.X.mean():.2f}")

## Spatial coordinates

In [ ]:
coords = adata.obsm['spatial']
print(f"Shape: {coords.shape}")
print(f"X range: {coords[:, 0].min():.1f} to {coords[:, 0].max():.1f} µm")
print(f"Y range: {coords[:, 1].min():.1f} to {coords[:, 1].max():.1f} µm")

## Distance columns

In [ ]:
# List all distance columns stored in obs
dist_cols = [c for c in adata.obs.columns if 'distance' in c]
print(f"{len(dist_cols)} distance columns:")
print(dist_cols)

In [ ]:
# Signed distances to Tumour and Stroma
signed_cols = [c for c in dist_cols if 'signed' in c]
adata.obs[signed_cols].describe().round(2)

In [ ]:
# Distance to CD8 cells
cd8_dist_col = [c for c in dist_cols if 'cd8_cells' in c and 'detection' in c]
if cd8_dist_col:
    print(adata.obs[cd8_dist_col[0]].describe().round(2))

## Quick sanity checks

In [ ]:
# NaN counts across obs metadata columns
nan_counts = adata.obs.isna().sum()
nan_counts[nan_counts > 0].sort_values(ascending=False)

In [ ]:
# NaN counts in X (marker matrix)
nan_in_X = np.isnan(adata.X).sum(axis=0)
for name, count in zip(adata.var_names, nan_in_X):
    if count > 0:
        print(f"  {name}: {count:,} NaN values")
if nan_in_X.sum() == 0:
    print("No NaN values in marker matrix ✓")